# Course 2 — Data Manipulation (Pandas)

Live-coding notebook that mirrors the slide deck.
Concrete examples lifted from Aurélien Géron's Pandas tutorial.

**Sections**
1. Series & DataFrames (0:00–0:30)
2. Select & Filter (0:30–1:00)
3. Clean, Group, Join (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
import pandas as pd
import numpy as np
pd.__version__


## 1. Series & DataFrames

### Series — your first one

In [ ]:
s = pd.Series([2, -1, 3, 5])
s


### NumPy plus labels

In [ ]:
np.exp(s)


In [ ]:
s + [1000, 2000, 3000, 4000]


In [ ]:
s + 1000      # scalar broadcasts


In [ ]:
s < 0         # element-wise comparison


### Index labels — name your rows

In [ ]:
s2 = pd.Series([68, 83, 112, 68], index=['alice', 'bob', 'charles', 'darwin'])
s2


In [ ]:
print(s2['bob'])
print(s2.loc['bob'])
print(s2.iloc[1])


### Gotcha — default-int labels are *labels*, not positions

In [ ]:
surprise = pd.Series([1000, 1001, 1002, 1003])
slice_ = surprise[2:]
print(slice_)

try:
    slice_[0]
except KeyError as e:
    print('Key error:', e)

print('via iloc:', slice_.iloc[0])


### Build from a dict

In [ ]:
weights = {'alice': 68, 'bob': 83, 'colin': 86, 'darwin': 68}
s3 = pd.Series(weights)
s3


### Automatic alignment

In [ ]:
print(s2.keys())
print(s3.keys())
s2 + s3


### Build a DataFrame — from Series

In [ ]:
people_dict = {
    'weight':    pd.Series([68, 83, 112], index=['alice', 'bob', 'charles']),
    'birthyear': pd.Series([1984, 1985, 1992],
                           index=['bob', 'alice', 'charles'], name='year'),
    'children':  pd.Series([0, 3], index=['charles', 'bob']),
    'hobby':     pd.Series(['Biking', 'Dancing'], index=['alice', 'bob']),
}
people = pd.DataFrame(people_dict)
people


### Build a DataFrame — from values + labels

In [ ]:
values = [[1985, np.nan, 'Biking',  68],
          [1984, 3,      'Dancing', 83],
          [1992, 0,       np.nan,  112]]
d3 = pd.DataFrame(values,
                  columns=['birthyear', 'children', 'hobby', 'weight'],
                  index=['alice', 'bob', 'charles'])
d3


### The first three things on any fresh dataset

In [ ]:
people.head()


In [ ]:
people.info()


In [ ]:
people.describe(include='all')


## 2. Select & Filter

### Columns

In [ ]:
people['birthyear']


In [ ]:
people[['birthyear', 'hobby']]


### Rows — three ways

In [ ]:
people.loc['charles']


In [ ]:
people.iloc[2]


In [ ]:
people.iloc[1:3]


In [ ]:
people[np.array([True, False, True])]


In [ ]:
people[people['birthyear'] < 1990]


### Boolean masks compose with `&` / `|` / `~`

In [ ]:
people[(people['birthyear'] < 1990) & (people['weight'] > 70)]


In [ ]:
people[people['hobby'].isin(['Biking', 'Dancing'])]


In [ ]:
people[~people['hobby'].isna()]


### Cleaner: `.query`

In [ ]:
people['age'] = 2025 - people['birthyear']
people['pets'] = pd.Series({'bob': 0, 'charles': 5})
people.query('age > 30 and pets == 0')


### Add & drop columns

In [ ]:
people['over 30'] = people['age'] > 30
people.insert(1, 'height', [172, 181, 185])
people


### `.assign` — chainable feature engineering

In [ ]:
(people
   .assign(bmi      = lambda df: df['weight'] / (df['height'] / 100) ** 2)
   .assign(overwt   = lambda df: df['bmi'] > 25)
)


### Sorting

In [ ]:
people.sort_values(by='age')


## 3. Clean, Group, Join

### Build two frames with mismatched labels

In [ ]:
grades_array = np.array([[8, 8, 9], [10, 9, 9], [4, 8, 2], [9, 10, 10]])
grades = pd.DataFrame(grades_array,
                      columns=['sep', 'oct', 'nov'],
                      index=['alice', 'bob', 'charles', 'darwin'])
grades


In [ ]:
bonus = pd.DataFrame(
    [[0, np.nan, 2], [np.nan, 1, 0], [0, 1, 0], [3, 3, 0]],
    columns=['oct', 'nov', 'dec'],
    index=['bob', 'colin', 'darwin', 'charles'])
bonus


In [ ]:
(grades + bonus)         # aligned by label; missing pairs -> NaN


### Fill vs drop

In [ ]:
(grades + bonus).fillna(0)


In [ ]:
(grades + bonus).dropna(how='all')


### Interpolate

In [ ]:
bonus.interpolate(axis=1)


### groupby — the workhorse

In [ ]:
final = (grades + bonus).fillna(0)
final['hobby'] = ['Biking', 'Dancing', np.nan, 'Dancing', 'Biking']
final


In [ ]:
final.groupby('hobby').mean(numeric_only=True)   # NaN groups skipped


In [ ]:
final.groupby('hobby').agg(
    n      = ('sep', 'size'),
    sep_mu = ('sep', 'mean'),
    nov_std= ('nov', 'std'),
)


### transform — aggregate but keep original shape

In [ ]:
final['nov_z'] = (
    final.groupby('hobby')['nov']
         .transform(lambda x: (x - x.mean()) / x.std())
)
final


### Pivot tables

In [ ]:
more = final.reset_index().melt(id_vars=['index', 'hobby'],
                                value_vars=['sep', 'oct', 'nov'],
                                var_name='month', value_name='grade')
more = more.rename(columns={'index': 'name'})
more.head()


In [ ]:
pd.pivot_table(more, index='name', values='grade',
               columns='month', margins=True)


### SQL-style joins

In [ ]:
city_loc = pd.DataFrame(
    [['CA', 'San Francisco', 37.78, -122.42],
     ['NY', 'New York',      40.71,  -74.01],
     ['FL', 'Miami',         25.79,  -80.32]],
    columns=['state', 'city', 'lat', 'lng'])

city_pop = pd.DataFrame(
    [[ 808976, 'San Francisco', 'California'],
     [8363710, 'New York',      'New-York'],
     [2242193, 'Houston',       'Texas']],
    columns=['population', 'city', 'state'])

pd.merge(city_loc, city_pop, on='city')               # INNER


In [ ]:
pd.merge(city_loc, city_pop, on='city', how='outer')  # OUTER


### Concatenation

In [ ]:
pd.concat([city_loc, city_pop], ignore_index=True)


### Time series — one minute tour

In [ ]:
temps = [4.4, 5.1, 6.1, 6.2, 6.1, 6.1, 5.7, 5.2, 4.7, 4.1, 3.9, 3.5]
dates = pd.date_range('2016/10/29 5:30pm', periods=12, freq='h')
ts = pd.Series(temps, index=dates, name='Temperature')
ts.head()


In [ ]:
ts.resample('2h').mean()


In [ ]:
ts.resample('15min').interpolate(method='cubic').head()


### Recap
* Series = labeled NumPy array. DataFrame = dict of aligned Series.
* `.loc` for labels, `.iloc` for positions, masks & `.query` for filters.
* Group, aggregate, derive — in chained pipelines.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

# Exercise 1 — DataFrame I/O & Inspection

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd


**Task 1.** Load the `tips` dataset. Show the first 5 rows and the dtypes.

In [ ]:
# your code here


**Task 2.** How many rows and columns does it have? How many numeric columns?

In [ ]:
# your code here


**Task 3.** Print a 5-number summary (`describe()`) for the numeric columns only.

In [ ]:
# your code here


**Task 4.** Set the index to a new column `bill_id` (just a 0-based range).

In [ ]:
# your code here


### Exercise 1 — Solution

# Exercise 1 — Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
df = load_dataset('tips')


**Task 1.**

In [ ]:
print(df.head())
print(df.dtypes)


**Task 2.**

In [ ]:
print('shape:', df.shape)
print('numeric cols:', df.select_dtypes('number').columns.tolist())


**Task 3.**

In [ ]:
df.describe()


**Task 4.**

In [ ]:
df = df.assign(bill_id=range(len(df))).set_index('bill_id')
df.head()


---

## Exercise 2

# Exercise 2 — Selection & Filtering

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
df = load_dataset('tips')


**Task 1.** Select rows 10–19 (inclusive) and only the columns `total_bill`, `tip`, `day`. Use `.loc`.

In [ ]:
# your code here


**Task 2.** Find all dinners where the tip was at least 20% of the bill.

In [ ]:
# your code here


**Task 3.** Same as Task 2 but use `df.query(...)`.

In [ ]:
# your code here


**Task 4.** What is the average tip on weekends (Sat/Sun) vs weekdays?

In [ ]:
# your code here


### Exercise 2 — Solution

# Exercise 2 — Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
df = load_dataset('tips')


**Task 1.**

In [ ]:
df.loc[10:19, ['total_bill', 'tip', 'day']]


**Task 2.**

In [ ]:
mask = (df['time'] == 'Dinner') & (df['tip'] / df['total_bill'] >= 0.20)
df[mask].head()


**Task 3.**

In [ ]:
df.query('time == "Dinner" and tip / total_bill >= 0.20').head()


**Task 4.**

In [ ]:
(df.assign(weekend=df['day'].isin(['Sat', 'Sun']))
   .groupby('weekend')['tip'].mean().round(2))


---

## Exercise 3

# Exercise 3 — Cleaning & Grouping

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
df = load_dataset('penguins')


**Task 1.** Report the number of NaNs per column.

In [ ]:
# your code here


**Task 2.** Drop every row that has at least one NaN. How many rows did you lose?

In [ ]:
# your code here


**Task 3.** On the cleaned data: per-species mean and std of `body_mass_g`. Round to 1 decimal.

In [ ]:
# your code here


**Task 4.** Add a column `mass_kg` (= body_mass_g / 1000) and a column `bill_ratio` (= bill_length / bill_depth). Sort by `bill_ratio` descending and show the top 5.

In [ ]:
# your code here


### Exercise 3 — Solution

# Exercise 3 — Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
df = load_dataset('penguins')


**Task 1.**

In [ ]:
df.isna().sum()


**Task 2.**

In [ ]:
clean = df.dropna()
print('lost', len(df) - len(clean), 'rows')


**Task 3.**

In [ ]:
clean.groupby('species')['body_mass_g'].agg(['mean', 'std']).round(1)


**Task 4.**

In [ ]:
clean = clean.assign(
    mass_kg=clean['body_mass_g'] / 1000,
    bill_ratio=clean['bill_length_mm'] / clean['bill_depth_mm'],
)
clean.sort_values('bill_ratio', ascending=False).head()
